# CSC-114 Final Project
## Network Intrusion Detection — Regression Classification with NSL-KDD
**Joe Philippe | FTCC CSC-114 | Summer 2026**

---
**Project Goal:** Train a Keras dense neural network using sigmoid (regression classification) output to predict the probability that a network connection is an attack. Tune the model by adjusting epoch count (5, 10, 20, 40) and record accuracy and loss at each setting.

**Dataset:** NSL-KDD — cleaned network intrusion detection dataset, 125,973 training records, 22,544 test records.

**Tuning Variable:** Number of epochs only (5 → 10 → 20 → 40)

---
## Cell 1 — Environment Setup
Install required libraries. Verify GPU is active: **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install tensorflow pandas scikit-learn matplotlib --quiet

import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:',  len(tf.config.list_physical_devices('GPU')) > 0)

**Expected output:**
```
TensorFlow version: 2.x.x
GPU available: True
```

---
## Cell 2 — Load the NSL-KDD Dataset
Fetch train and test CSVs directly from a public GitHub mirror. No Kaggle account or API key required.

In [ ]:
import pandas as pd

COLS = [
    'duration','protocol_type','service','flag','src_bytes',
    'dst_bytes','land','wrong_fragment','urgent','hot',
    'num_failed_logins','logged_in','num_compromised','root_shell',
    'su_attempted','num_root','num_file_creations','num_shells',
    'num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate',
    'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count',
    'dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate',
    'dst_host_srv_rerror_rate','label','difficulty'
]

BASE = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/'
df_train = pd.read_csv(BASE + 'KDDTrain+.txt', names=COLS)
df_test  = pd.read_csv(BASE + 'KDDTest+.txt',  names=COLS)

print('Train shape:', df_train.shape)
print('Test shape: ', df_test.shape)
print()
print('Label value counts (train):')
print(df_train['label'].value_counts().head(10))

**Expected output:**
```
Train shape: (125973, 43)
Test shape:  (22544, 43)
```

---
## Cell 3 — Exploratory Data Chart
Plot class distribution: normal vs. attack records in the training set.
**Save the chart image — it goes in the README.**

In [ ]:
import matplotlib.pyplot as plt

counts = df_train['label'].apply(
    lambda x: 'normal' if x == 'normal' else 'attack'
).value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values,
              color=['steelblue', 'crimson'], edgecolor='black')

# label each bar with its count
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 800,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('NSL-KDD Training Set — Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Record Count')
ax.set_xlabel('Label')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print('Chart saved: class_distribution.png')

**Expected output:** Bar chart showing normal and attack record counts with values labeled on each bar.

---
## Cell 4 — Preprocessing
- One-hot encode categorical columns: `protocol_type`, `service`, `flag`
- Convert label to binary: 0 = normal, 1 = attack
- Normalize all numeric features to 0–1 range with MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

def preprocess(df):
    df = df.copy()
    # binary label: 0 = normal, 1 = attack
    df['label'] = (df['label'] != 'normal').astype(int)
    # one-hot encode categorical columns
    df = pd.get_dummies(df, columns=['protocol_type', 'service', 'flag'])
    # drop difficulty column (not a feature)
    df = df.drop(columns=['difficulty'])
    return df

train = preprocess(df_train)
# align test columns to train to handle any missing one-hot categories
test  = preprocess(df_test).reindex(columns=train.columns, fill_value=0)

X_train = train.drop(columns=['label']).values.astype('float32')
y_train = train['label'].values
X_test  = test.drop(columns=['label']).values.astype('float32')
y_test  = test['label'].values

# normalize features to 0-1 range
scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print('Features after one-hot encoding:', X_train.shape[1])
print('Attack rate in training set:',
      f'{y_train.mean()*100:.1f}%')

**Expected output:**
```
X_train shape: (125973, ~122)
Features after one-hot encoding: ~122
```

---
## Cell 5 — Build the Model
Dense neural network with **sigmoid output** — this is the regression classification architecture.

- `Dense(1, activation='sigmoid')` → outputs a probability score 0.0–1.0
- Loss: `binary_crossentropy`
- Optimizer: `adam`

In [ ]:
from tensorflow import keras

def build_model(input_dim):
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation='sigmoid')  # regression classification output
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# preview the architecture
model = build_model(X_train.shape[1])
model.summary()

**Expected output:** Model summary table showing three Dense layers and two Dropout layers with parameter counts.

---
## Cell 6 — Epoch Experiment Loop
Train the model at **4 epoch counts: 5, 10, 20, 40**.
Each run starts from a fresh model (same architecture, new weights) for a fair comparison.
Records train accuracy, validation accuracy, and validation loss.

In [ ]:
results = []
EPOCH_LIST = [5, 10, 20, 40]

print(f'{'Epochs':>8} | {'Train Acc':>10} | {'Val Acc':>10} | {'Val Loss':>10}')
print('-' * 48)

for epochs in EPOCH_LIST:
    m = build_model(X_train.shape[1])
    history = m.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=512,
        validation_split=0.1,
        verbose=0
    )
    train_acc = history.history['accuracy'][-1]
    val_acc   = history.history['val_accuracy'][-1]
    val_loss  = history.history['val_loss'][-1]

    results.append({
        'epochs':    epochs,
        'train_acc': train_acc,
        'val_acc':   val_acc,
        'val_loss':  val_loss
    })
    print(f'{epochs:>8} | {train_acc:>10.4f} | {val_acc:>10.4f} | {val_loss:>10.4f}')

print()
print('Epoch experiment complete.')

**Expected output:** One row per epoch count. Val Loss should decrease then rise again at higher epochs (overfitting signal).
```
  Epochs |  Train Acc |    Val Acc |   Val Loss
------------------------------------------------
       5 |     0.xxxx |     0.xxxx |     0.xxxx
      10 |     0.xxxx |     0.xxxx |     0.xxxx
      20 |     0.xxxx |     0.xxxx |     0.xxxx
      40 |     0.xxxx |     0.xxxx |     0.xxxx
```

---
## Cell 7 — Epoch Results Chart
Dual-axis line chart: accuracy (blue) and validation loss (red) plotted against epoch count.
**Save this chart — it goes in the README.**

In [ ]:
res_df = pd.DataFrame(results)

fig, ax1 = plt.subplots(figsize=(8, 5))

# accuracy lines on left axis
ax1.plot(res_df['epochs'], res_df['val_acc'],
         'b-o', linewidth=2, markersize=8, label='Val Accuracy')
ax1.plot(res_df['epochs'], res_df['train_acc'],
         'b--o', linewidth=2, markersize=8, label='Train Accuracy')
ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('Accuracy', color='blue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_ylim(0.7, 1.01)
ax1.set_xticks(res_df['epochs'])

# validation loss on right axis
ax2 = ax1.twinx()
ax2.plot(res_df['epochs'], res_df['val_loss'],
         'r-s', linewidth=2, markersize=8, label='Val Loss')
ax2.set_ylabel('Validation Loss', color='red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Epoch Tuning — NSL-KDD Intrusion Detection',
          fontsize=13, fontweight='bold')

# combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('epoch_tuning.png', dpi=150)
plt.show()
print('Chart saved: epoch_tuning.png')

# print results table
print()
print('Results summary:')
print(res_df.to_string(index=False))

**Expected output:** Dual-axis chart saved as `epoch_tuning.png`. The val accuracy curve peaks and the val loss curve bottoms out at the optimal epoch count — that is the best setting for Cell 8.

---
## Cell 8 — Final Evaluation and Predictions
Retrain at the best epoch count from Cell 7. Run predictions on the test set.
Output shows **raw probability scores** — demonstrating regression classification output.

In [ ]:
# ── UPDATE this value after reviewing Cell 7 results ──
BEST_EPOCHS = 20
# ──────────────────────────────────────────────────────

print(f'Training final model for {BEST_EPOCHS} epochs...')
final_model = build_model(X_train.shape[1])
final_model.fit(
    X_train, y_train,
    epochs=BEST_EPOCHS,
    batch_size=512,
    verbose=0
)

# predict probabilities on test set
probs = final_model.predict(X_test, verbose=0).flatten()
preds = (probs >= 0.5).astype(int)

print()
print('Sample predictions — regression classification output:')
print(f'  {"Probability":>12} | {"Predicted":>10} | {"Actual":>8}')
print('  ' + '-' * 38)
for i in range(10):
    label_pred = 'attack' if preds[i] == 1 else 'normal'
    label_true = 'attack' if y_test[i] == 1 else 'normal'
    match = '✓' if preds[i] == y_test[i] else '✗'
    print(f'  {probs[i]:>12.4f} | {label_pred:>10} | {label_true:>8}  {match}')

# final test set evaluation
loss, acc = final_model.evaluate(X_test, y_test, verbose=0)
print()
print('=' * 42)
print(f'  Final Test Accuracy : {acc:.4f} ({acc*100:.2f}%)')
print(f'  Final Test Loss     : {loss:.4f}')
print(f'  Best Epoch Setting  : {BEST_EPOCHS}')
print('=' * 42)

**Expected output:** 10 sample connections with their probability score, predicted label, and true label. Final accuracy and loss on the full 22,544-record test set.

---
## Project Summary

| Item | Value |
|---|---|
| Dataset | NSL-KDD Network Intrusion |
| Model type | Dense neural network (sigmoid output) |
| Output | Probability score 0.0–1.0 |
| Loss function | Binary cross-entropy |
| Tuning variable | Number of epochs |
| Epoch values tested | 5, 10, 20, 40 |
| Best epoch setting | *Update after running Cell 7* |

**Charts produced:**
- `class_distribution.png` — Cell 3
- `epoch_tuning.png` — Cell 7

*CSC-114 Artificial Intelligence I | Fayetteville Technical Community College | Summer 2026*